In [1]:
# Install required libraries
!pip install dash plotly pandas

   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   -------------- ------------------------- 2.6/7.2 MB 11.5 MB/s eta 0:00:01
   -------------------------- ------------- 4.7/7.2 MB 11.1 MB/s eta 0:00:01
   ---------------------------------------  7.1/7.2 MB 11.3 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 10.7 MB/s  0:00:00

   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   ---

In [2]:
dashboard_code = '''
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# Load the data
df = pd.read_csv("outputs/final_analysis_with_projections.csv")

# Initialize the Dash app
app = dash.Dash(__name__)

# Get list of areas for dropdown
area_list = sorted(df["Area name"].unique())

# Define colors for status
status_colors = {
    "On Track (Improving)": "#2ecc71",
    "Borderline (Slow Decline)": "#f39c12", 
    "Off Track (Rapid Decline)": "#e74c3c"
}

# App layout
app.layout = html.Div([
    html.H1("England Health Inequality Dashboard", 
            style={"textAlign": "center", "color": "#2c3e50", "padding": "20px"}),
    
    html.Div([
        html.H3("Tracking Progress Toward 2035 Health Equity Target", 
                style={"textAlign": "center", "color": "#7f8c8d"})
    ]),
    
    # Summary statistics
    html.Div([
        html.Div([
            html.H4("128", style={"color": "#2c3e50", "margin": "0"}),
            html.P("Areas Analyzed", style={"color": "#7f8c8d"})
        ], style={"textAlign": "center", "padding": "20px", "flex": "1"}),
        
        html.Div([
            html.H4("18", style={"color": "#2ecc71", "margin": "0"}),
            html.P("Improving (14%)", style={"color": "#7f8c8d"})
        ], style={"textAlign": "center", "padding": "20px", "flex": "1"}),
        
        html.Div([
            html.H4("110", style={"color": "#e74c3c", "margin": "0"}),
            html.P("Worsening (86%)", style={"color": "#7f8c8d"})
        ], style={"textAlign": "center", "padding": "20px", "flex": "1"}),
        
        html.Div([
            html.H4("+2.6", style={"color": "#e74c3c", "margin": "0"}),
            html.P("Average Change (years)", style={"color": "#7f8c8d"})
        ], style={"textAlign": "center", "padding": "20px", "flex": "1"}),
    ], style={"display": "flex", "justifyContent": "space-around", "backgroundColor": "#ecf0f1", 
              "borderRadius": "10px", "margin": "20px"}),
    
    # Status distribution chart
    html.Div([
        dcc.Graph(id="status-chart")
    ], style={"padding": "20px"}),
    
    # Area comparison tool
    html.Div([
        html.H3("Compare Local Authorities", style={"color": "#2c3e50"}),
        html.Div([
            html.Div([
                html.Label("Select Area 1:"),
                dcc.Dropdown(
                    id="area1-dropdown",
                    options=[{"label": area, "value": area} for area in area_list],
                    value="Birmingham"
                )
            ], style={"width": "45%", "display": "inline-block"}),
            
            html.Div([
                html.Label("Select Area 2:"),
                dcc.Dropdown(
                    id="area2-dropdown",
                    options=[{"label": area, "value": area} for area in area_list],
                    value="Richmond upon Thames"
                )
            ], style={"width": "45%", "display": "inline-block", "marginLeft": "5%"}),
        ]),
        
        dcc.Graph(id="comparison-chart")
    ], style={"padding": "20px", "backgroundColor": "#f8f9fa", "borderRadius": "10px", "margin": "20px"}),
    
    # Footer
    html.Div([
        html.P("Data: ONS Health State Life Expectancy 2011-2024 | Analysis: Python (pandas, plotly)",
               style={"textAlign": "center", "color": "#95a5a6", "fontSize": "12px"})
    ], style={"padding": "20px"})
])

# Callback for status chart
@app.callback(
    Output("status-chart", "figure"),
    Input("status-chart", "id")
)
def update_status_chart(_):
    status_counts = df["status_2035"].value_counts()
    
    fig = go.Figure(data=[
        go.Bar(
            x=["On Track<br>(Improving)", "Borderline<br>(Slow Decline)", "Off Track<br>(Rapid Decline)"],
            y=[
                status_counts.get("On Track (Improving)", 0),
                status_counts.get("Borderline (Slow Decline)", 0),
                status_counts.get("Off Track (Rapid Decline)", 0)
            ],
            marker_color=["#2ecc71", "#f39c12", "#e74c3c"],
            text=[
                status_counts.get("On Track (Improving)", 0),
                status_counts.get("Borderline (Slow Decline)", 0),
                status_counts.get("Off Track (Rapid Decline)", 0)
            ],
            textposition="outside",
            textfont=dict(size=14, color="black")
        )
    ])
    
    fig.update_layout(
        title="2035 Outlook: Only 14% of Areas Are Improving",
        yaxis_title="Number of Local Authorities",
        height=400,
        showlegend=False
    )
    
    return fig

# Callback for comparison chart
@app.callback(
    Output("comparison-chart", "figure"),
    [Input("area1-dropdown", "value"),
     Input("area2-dropdown", "value")]
)
def update_comparison(area1, area2):
    # Get data for both areas
    data1 = df[df["Area name"] == area1].iloc[0]
    data2 = df[df["Area name"] == area2].iloc[0]
    
    fig = go.Figure()
    
    # Area 1
    fig.add_trace(go.Bar(
        name=area1,
        x=["2011-2013", "2022-2024", "2035 Projected"],
        y=[data1["gap_2011_2013"], data1["gap_2022_2024"], data1["gap_2035_projected"]],
        marker_color="#3498db"
    ))
    
    # Area 2
    fig.add_trace(go.Bar(
        name=area2,
        x=["2011-2013", "2022-2024", "2035 Projected"],
        y=[data2["gap_2011_2013"], data2["gap_2022_2024"], data2["gap_2035_projected"]],
        marker_color="#e67e22"
    ))
    
    fig.update_layout(
        title=f"HLE Gap Comparison: {area1} vs {area2}",
        yaxis_title="HLE Gap (years in poor health)",
        barmode="group",
        height=400
    )
    
    return fig

# Run the app
if __name__ == "__main__":
    print("\\n" + "="*60)
    print("DASHBOARD STARTING...")
    print("="*60)
    print("\\nOpen your browser and go to: http://127.0.0.1:8050")
    print("\\nPress CTRL+C to stop the dashboard\\n")
    app.run_server(debug=True, use_reloader=False)
'''

# Save the dashboard code to a file
with open('dashboard_app.py', 'w', encoding='utf-8') as f:
    f.write(dashboard_code)

print("✓ Dashboard file created: dashboard_app.py")
print("\nNext step: Run the dashboard!")

✓ Dashboard file created: dashboard_app.py

Next step: Run the dashboard!


In [3]:
import shutil
import os

# Source (where the file was created)
source = 'C:/Users/Aswathy Harikumar/Documents/hle-gap-project/dashboard_app.py'

# Check if it exists
if os.path.exists(source):
    print(f"✓ File found at: {source}")
else:
    print(f"✗ File NOT found at: {source}")
    # Try to find it
    possible_locations = [
        'C:/Users/Aswathy Harikumar/OneDrive/Documents/hle-gap-project/dashboard_app.py',
        'C:/Users/Aswathy Harikumar/dashboard_app.py',
        'dashboard_app.py'
    ]
    for loc in possible_locations:
        if os.path.exists(loc):
            print(f"✓ Found it at: {loc}")
            source = loc
            break
        

✗ File NOT found at: C:/Users/Aswathy Harikumar/Documents/hle-gap-project/dashboard_app.py
✓ Found it at: dashboard_app.py


In [4]:
import os
import shutil

# Where are we now?
current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

# Where the file currently is
source = os.path.join(current_dir, 'dashboard_app.py')
print(f"File is at: {source}")

# Where we want it
destination = 'C:/Users/Aswathy Harikumar/Documents/hle-gap-project/dashboard_app.py'

# Copy it there
shutil.copy(source, destination)
print(f"\n✓ Copied to: {destination}")

Current directory: C:\Users\Aswathy Harikumar\Documents\hle-gap-project\notebooks
File is at: C:\Users\Aswathy Harikumar\Documents\hle-gap-project\notebooks\dashboard_app.py

✓ Copied to: C:/Users/Aswathy Harikumar/Documents/hle-gap-project/dashboard_app.py


In [6]:
# Read the current file
with open('C:/Users/Aswathy Harikumar/Documents/hle-gap-project/dashboard_app.py', 'r') as f:
    content = f.read()

# Replace the old command with the new one
content = content.replace('app.run_server(debug=True, use_reloader=False)', 
                          'app.run(debug=True)')

# Save it back
with open('C:/Users/Aswathy Harikumar/Documents/hle-gap-project/dashboard_app.py', 'w') as f:
    f.write(content)

print("✓ Dashboard fixed!")

✓ Dashboard fixed!


In [7]:
import os
os.chdir('C:/Users/Aswathy Harikumar/Documents/hle-gap-project')

# Create requirements.txt (tells Render what to install)
requirements = """dash==4.0.0
plotly==5.24.1
pandas==2.2.3
gunicorn==23.0.0
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("✓ requirements.txt created")

# Create Procfile (tells Render how to run the app)
procfile = """web: gunicorn dashboard_app:server
"""

with open('Procfile', 'w') as f:
    f.write(procfile)

print("✓ Procfile created")

✓ requirements.txt created
✓ Procfile created


In [8]:
# Update dashboard_app.py for deployment
with open('dashboard_app.py', 'r') as f:
    content = f.read()

# Add server variable (required for deployment)
content = content.replace(
    'app = dash.Dash(__name__)',
    'app = dash.Dash(__name__)\nserver = app.server  # Required for deployment'
)

with open('dashboard_app.py', 'w') as f:
    f.write(content)

print("✓ Dashboard updated for deployment")

✓ Dashboard updated for deployment


In [9]:
# Read the current dashboard
with open('C:/Users/Aswathy Harikumar/Documents/hle-gap-project/dashboard_app.py', 'r') as f:
    content = f.read()

# Fix the title
content = content.replace(
    'html.H1("England Health Inequality Dashboard"',
    'html.H1("England Healthy Life Expectancy Gap Dashboard"'
)

# Fix the subtitle
content = content.replace(
    'html.H3("Tracking Progress Toward 2035 Health Equity Target"',
    'html.H3("Measuring Years Lived in Poor Health: 2011-2024 Trends & 2035 Projections"'
)

# Fix the status chart title
content = content.replace(
    'fig.update_layout(title="Status Distribution"',
    'fig.update_layout(title="HLE Gap Trends (2011-2024): Are Areas Improving or Worsening?"'
)

# Fix the comparison chart title
content = content.replace(
    'fig.update_layout(title=f"{a1} vs {a2}"',
    'fig.update_layout(title=f"HLE Gap Comparison: Years in Poor Health - {a1} vs {a2}"'
)

# Add explanation text
content = content.replace(
    'html.H4("128 Areas | 18 Improving (14%) | 110 Worsening (86%) | Average: +2.6 years"',
    '''html.Div([
        html.P("The HLE Gap measures how many years people spend in poor health (Life Expectancy - Healthy Life Expectancy)", 
               style={"textAlign": "center", "fontSize": "14px", "color": "#555", "marginBottom": "5px"}),
        html.H4("128 Areas Analyzed | 18 Improving (gap shrinking) | 110 Worsening (gap widening) | Average Change: +2.6 years worse",
                style={"textAlign": "center", "color": "#e74c3c", "fontWeight": "bold"})
    ])'''
)

# Save the updated file
with open('C:/Users/Aswathy Harikumar/Documents/hle-gap-project/dashboard_app.py', 'w') as f:
    f.write(content)

print("✓ Dashboard labels improved!")
print("\nGo to Command Prompt and run: python dashboard_app.py")
print("Then refresh your browser to see the changes")

✓ Dashboard labels improved!

Go to Command Prompt and run: python dashboard_app.py
Then refresh your browser to see the changes
